# NB01 — Exploration du corpus de peptides

**Semaine 1 — peplib v0.1.** Ce notebook montre le pipeline complet du corpus :
chargement, filtrage, composition, divergence JS, descripteurs physico-chimiques,
PWM des extrémités, figures et redondance.

**Règle du projet :** les cellules n'appellent que `peplib` — aucune logique
métier (formules, filtrage, génération de figures) n'est réimplémentée ici.
Tous les résultats sont régénérables depuis `data/raw/` par le code de `peplib/`.

**Limites à garder en tête :** les contraintes structurales restent des proxies
(hélicité, amphipathicité HP) et les scores ne sont **pas** des mesures
d'activité biologique.

In [ ]:
import pandas as pd

import peplib.dedup
import peplib.filters
import peplib.stats
import peplib.descriptors
import peplib.pwm
import peplib.figures

In [ ]:
corpus = peplib.figures.load_corpus()
print({s: len(df) for s, df in corpus.items()})
corpus["uniprot"].head()

In [ ]:
# T11–T14 : table de rejets produite par le pipeline de filtrage
# (chaque séquence rejetée comptée dans sa PREMIÈRE cause).
rejets = pd.read_csv("data/processed/rejection_table_t14.csv")
rejets

In [ ]:
# T16 : composition en acides aminés f(a) par source (somme == 1).
comp = pd.read_csv("data/processed/composition_t16.csv", index_col=0)
comp

In [ ]:
# T17 : divergence Jensen–Shannon entre sources (nats, bornée par ln 2).
js = pd.read_csv("data/processed/js_divergence_t17.csv", index_col=0)
js

In [ ]:
# T18 : descripteurs physico-chimiques sur un sous-échantillon
# (200 premières séquences par source) pour un aperçu rapide.
parties = []
for s, df in corpus.items():
    d = peplib.figures.compute_descriptors(df.head(200))
    d["source"] = s
    parties.append(d)
desc = pd.concat(parties, ignore_index=True)
desc.describe()

In [ ]:
# T20 : PWM alignée depuis N et C (jamais tronquée), + information par
# position avec correction de petit échantillon e(n) (Schneider & Stephens).
pwms = peplib.pwm.run_pwm_pipeline()
print(list(pwms.keys()))
info_n = pd.read_csv("results/information_N.csv", index_col=0)
info_c = pd.read_csv("results/information_C.csv", index_col=0)
info_n.head(10)

In [ ]:
# T21 : figures régénérables — distributions des descripteurs,
# heatmap PWM N/C, courbe d'information. Les PNG sont dans results/.
chemins = peplib.figures.run_figures_pipeline()
chemins

In [ ]:
# T22 et T25 : correspondances exactes par source, puis quasi-doublons à 90 %.
parties = []
for s, df in corpus.items():
    r = peplib.dedup.exact_dedup_stats(df["sequence"])
    r["scope"] = s
    parties.append(r)
redondance = pd.DataFrame(parties).set_index("scope")
print(redondance)

quasi90 = pd.read_csv("data/processed/near_duplicates_t25.csv")
quasi90

## Conclusion

- La redondance exacte est forte (environ 27 %), d'où la nécessité du
  traitement T25 avant toute autre chose (division 90/5/5 par clusters au mois 2).
- La composition moyenne diffère entre sources (divergence JS > 0) sans être
  massive ; les descripteurs discriminants attendus restent la charge nette
  et le moment hydrophobe (signature amphipathique des AMP).
- La conservation des extrémités est visible dans l'information par position,
  en particulier à proximité du terminus C (à interpréter avec prudence :
  très peu de séquences couvrent les positions extrêmes).

**Rappel des bornes :** structure = proxies, scores ≠ activité mesurée ;
aucune affirmation biologique sans validation par laboratoire qualifié.